In [ ]:
%cd ..

In [ ]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import pandas as pd
import zarr
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from common.sample_db import SampleDB

db = SampleDB()

### distributiuon of losses from score

In [ ]:
MODEL = '232_keras/i9'
MODEL_NAME = MODEL.replace("/", "_").replace("_keras", "")

RUNS = [
    ("006", "sobol"),
    ("390", "importance_sampled"),
    ("500", "uniform_rnd")
]

BETA_STFT = 0.01

In [ ]:
dfs = []

for run, desc in RUNS:

    losses = db.losses_for(run=run, model=MODEL)
    df = pd.DataFrame(losses)
    df['desc'] = desc
    dfs.append(df)

losses_df = pd.concat(dfs)
losses_df.head()
#losses_1_df['src'] = 'sobol'
#losses_1_df.describe()

In [ ]:
desc_values = ['sobol', 'importance_sampled', 'uniform_rnd']
cols = ['loss', 'huber', 'stft']

# Keep histogram and mean-line colors aligned across both subplots.
palette = sns.color_palette(n_colors=len(desc_values))
palette_map = dict(zip(desc_values, palette))

shared_xlim = {
    col: (losses_df[col].min(), losses_df[col].max())
    for col in cols
}

fig, axes = plt.subplots(1, len(cols), figsize=(15, 6), squeeze=False)

for col_idx, col in enumerate(cols):
    ax = axes[0, col_idx]
    sns.histplot(
        data=losses_df,
        x=col,
        hue='desc',
        hue_order=desc_values,
        palette=palette_map,
        kde=True,
        stat='density',
        common_norm=False,
        alpha=0.35,
        element='step',
        ax=ax,
    )

    # Mean marker per desc in matching hue color.
    for desc in desc_values:
        desc_vals = losses_df.loc[losses_df['desc'] == desc, col].dropna()
        if not desc_vals.empty:
            ax.axvline(
                desc_vals.mean(),
                color=palette_map[desc],
                linestyle='--',
                linewidth=1.5,
                alpha=0.95,
            )

    ax.set_xlim(shared_xlim[col])
    ax.set_title(col)

fig.suptitle(f"run {MODEL_NAME}", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])

output_path = Path.home() / "Pictures" / f"is_losses_distributions.{MODEL_NAME}.jpg"
fig.savefig(output_path, format="jpg", dpi=200, bbox_inches="tight")
print(f"Saved: {output_path}")

plt.show()